# 🔍 检查 TS-SatFire 原始数据

查看 Google Drive 中已下载的原始数据文件夹结构和内容。

---

## Step 1: 挂载 Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path
import subprocess

# 挂载 Google Drive
drive.mount('/content/drive')

print("✅ Google Drive 挂载成功")

## Step 2: 检查原始数据文件夹

In [ ]:
# 设置数据路径
RAW_DATA_PATH = Path('/content/drive/MyDrive/FireST-GraphNet/ts-satfire_raw')

print("检查原始数据路径...\n")
print(f"路径: {RAW_DATA_PATH}")
print(f"存在: {RAW_DATA_PATH.exists()}")

if RAW_DATA_PATH.exists():
    print("\n✅ 找到原始数据文件夹！")
else:
    print("\n❌ 未找到原始数据文件夹")
    print("\n请检查路径是否正确，或者文件夹名称是否为：")
    print("  - ts-satfire_raw")
    print("  - ts-satfire-data")
    print("  - TS-SatFire")
    print("\n或者列出 FireST-GraphNet 目录下的所有文件夹：")
    parent = Path('/content/drive/MyDrive/FireST-GraphNet')
    if parent.exists():
        for item in sorted(parent.iterdir()):
            if item.is_dir():
                print(f"  📁 {item.name}")

## Step 3: 显示文件夹结构（第一层）

In [ ]:
if RAW_DATA_PATH.exists():
    print("=== 原始数据文件夹结构（第一层）===\n")
    
    items = sorted(RAW_DATA_PATH.iterdir())
    
    dirs = [item for item in items if item.is_dir()]
    files = [item for item in items if item.is_file()]
    
    print(f"总计: {len(dirs)} 个文件夹, {len(files)} 个文件\n")
    
    if dirs:
        print("📁 文件夹：")
        for i, d in enumerate(dirs[:20], 1):  # 只显示前20个
            # 统计子文件数量
            try:
                subfiles = list(d.iterdir())
                print(f"  {i:2d}. {d.name:40s} ({len(subfiles)} 项)")
            except:
                print(f"  {i:2d}. {d.name:40s} (无法访问)")
        
        if len(dirs) > 20:
            print(f"  ... 还有 {len(dirs) - 20} 个文件夹")
    
    if files:
        print("\n📄 文件：")
        for i, f in enumerate(files[:20], 1):  # 只显示前20个
            size_mb = f.stat().st_size / (1024**2)
            print(f"  {i:2d}. {f.name:40s} {size_mb:10.2f} MB")
        
        if len(files) > 20:
            print(f"  ... 还有 {len(files) - 20} 个文件")
else:
    print("⚠️  请先运行上面的单元格检查路径")

## Step 4: 统计数据类型和大小

In [ ]:
if RAW_DATA_PATH.exists():
    print("=== 数据统计 ===\n")
    
    # 按扩展名分类
    file_types = {}
    total_size = 0
    total_files = 0
    
    print("正在扫描文件...")
    
    for item in RAW_DATA_PATH.rglob('*'):
        if item.is_file():
            ext = item.suffix.lower() or '(无扩展名)'
            size = item.stat().st_size
            
            if ext not in file_types:
                file_types[ext] = {'count': 0, 'size': 0}
            
            file_types[ext]['count'] += 1
            file_types[ext]['size'] += size
            
            total_files += 1
            total_size += size
    
    print("\n📊 文件类型统计：\n")
    print(f"{'文件类型':<15} {'数量':>8} {'总大小':>12}")
    print("-" * 40)
    
    # 按大小排序
    for ext, info in sorted(file_types.items(), key=lambda x: x[1]['size'], reverse=True):
        size_gb = info['size'] / (1024**3)
        print(f"{ext:<15} {info['count']:>8} {size_gb:>10.2f} GB")
    
    print("-" * 40)
    print(f"{'总计':<15} {total_files:>8} {total_size / (1024**3):>10.2f} GB")
    
    print("\n💾 磁盘使用：")
    print(f"  总文件数: {total_files:,}")
    print(f"  总大小: {total_size / (1024**3):.2f} GB")
else:
    print("⚠️  请先运行上面的单元格检查路径")

## Step 5: 检查是否为 GeoTIFF 格式

In [ ]:
if RAW_DATA_PATH.exists():
    print("=== 检查 GeoTIFF 文件 ===\n")
    
    # 查找 .tif 或 .tiff 文件
    tif_files = list(RAW_DATA_PATH.rglob('*.tif')) + list(RAW_DATA_PATH.rglob('*.tiff'))
    
    if tif_files:
        print(f"✅ 找到 {len(tif_files)} 个 GeoTIFF 文件\n")
        
        # 按目录分组
        dirs_with_tif = {}
        for f in tif_files:
            parent = f.parent.name
            if parent not in dirs_with_tif:
                dirs_with_tif[parent] = []
            dirs_with_tif[parent].append(f.name)
        
        print("📁 包含 GeoTIFF 的文件夹：\n")
        for dirname, files in sorted(dirs_with_tif.items()):
            print(f"  {dirname}: {len(files)} 文件")
            # 显示前3个文件名作为示例
            for fname in sorted(files)[:3]:
                print(f"    - {fname}")
            if len(files) > 3:
                print(f"    ... 还有 {len(files) - 3} 个文件")
            print()
    else:
        print("⚠️  未找到 GeoTIFF 文件 (.tif/.tiff)")
        print("\n可能的原因：")
        print("1. 数据还在压缩包中（需要解压）")
        print("2. 文件扩展名不同")
        print("3. 数据格式不是 GeoTIFF")
else:
    print("⚠️  请先运行上面的单元格检查路径")

## Step 6: 查找特定数据类型（train/val/test）

In [ ]:
if RAW_DATA_PATH.exists():
    print("=== 查找 train/val/test 分割 ===\n")
    
    keywords = ['train', 'val', 'test', 'validation']
    
    found_splits = {}
    for keyword in keywords:
        # 查找包含关键词的文件夹
        matching_dirs = [d for d in RAW_DATA_PATH.rglob('*') if d.is_dir() and keyword in d.name.lower()]
        
        if matching_dirs:
            found_splits[keyword] = matching_dirs
    
    if found_splits:
        print("✅ 找到数据分割：\n")
        for keyword, dirs in found_splits.items():
            print(f"📂 {keyword.upper()}:")
            for d in dirs:
                rel_path = d.relative_to(RAW_DATA_PATH)
                try:
                    items = list(d.iterdir())
                    print(f"  - {rel_path} ({len(items)} 项)")
                except:
                    print(f"  - {rel_path} (无法访问)")
            print()
    else:
        print("⚠️  未找到明确的 train/val/test 分割")
        print("\n数据可能：")
        print("1. 在单个文件夹中（需要手动分割）")
        print("2. 使用不同的命名方式")
        print("3. 需要通过预处理脚本生成")
else:
    print("⚠️  请先运行上面的单元格检查路径")

## Step 7: 显示完整目录树（可选）

In [ ]:
if RAW_DATA_PATH.exists():
    print("=== 完整目录树（前2层）===\n")
    
    def print_tree(path, prefix="", max_depth=2, current_depth=0):
        """打印目录树"""
        if current_depth >= max_depth:
            return
        
        try:
            items = sorted(path.iterdir(), key=lambda x: (not x.is_dir(), x.name))
            
            for i, item in enumerate(items):
                is_last = i == len(items) - 1
                current_prefix = "└── " if is_last else "├── "
                
                if item.is_dir():
                    print(f"{prefix}{current_prefix}📁 {item.name}/")
                    
                    # 递归打印子目录
                    next_prefix = prefix + ("    " if is_last else "│   ")
                    print_tree(item, next_prefix, max_depth, current_depth + 1)
                else:
                    size_mb = item.stat().st_size / (1024**2)
                    if size_mb < 1:
                        size_str = f"{item.stat().st_size / 1024:.1f} KB"
                    else:
                        size_str = f"{size_mb:.1f} MB"
                    print(f"{prefix}{current_prefix}📄 {item.name} ({size_str})")
                
                # 限制显示数量
                if i >= 20:
                    remaining = len(items) - 20
                    print(f"{prefix}    ... 还有 {remaining} 项")
                    break
        
        except PermissionError:
            print(f"{prefix}    (无权限访问)")
    
    print(f"📁 {RAW_DATA_PATH.name}/")
    print_tree(RAW_DATA_PATH)
    
    print("\n💡 提示：如需查看更深层级，修改上面单元格中的 max_depth 参数")
else:
    print("⚠️  请先运行上面的单元格检查路径")

## Step 8: 检查数据是否适合预处理

In [ ]:
if RAW_DATA_PATH.exists():
    print("=== 预处理就绪检查 ===\n")
    
    checks = {
        "路径存在": RAW_DATA_PATH.exists(),
        "包含文件": len(list(RAW_DATA_PATH.rglob('*'))) > 0,
        "包含 GeoTIFF": len(list(RAW_DATA_PATH.rglob('*.tif')) + list(RAW_DATA_PATH.rglob('*.tiff'))) > 0,
    }
    
    print("✅ 检查项：\n")
    all_passed = True
    for check, passed in checks.items():
        status = "✅" if passed else "❌"
        print(f"  {status} {check}")
        if not passed:
            all_passed = False
    
    print("\n" + "="*50)
    
    if all_passed:
        print("\n🎉 数据看起来不错！可以开始预处理了！")
        print("\n下一步：")
        print("1. 克隆 TS-SatFire 预处理仓库")
        print("2. 运行 dataset_gen_afba.py 生成 AF/BA 数据")
        print("3. 运行 dataset_gen_pred.py 生成 Pred 数据")
        print("\n或者使用我们的自动化 notebook：")
        print("  data_tools/colab/colab_data_generation_and_training.ipynb")
    else:
        print("\n⚠️  数据可能有问题，请检查：")
        print("1. 是否已完整解压？")
        print("2. 路径是否正确？")
        print("3. 是否为 TS-SatFire 官方格式？")
else:
    print("⚠️  请先运行上面的单元格检查路径")

---

## 📝 总结

这个 notebook 帮助你：
- ✅ 验证原始数据是否正确下载和解压
- ✅ 查看文件夹结构和文件类型
- ✅ 统计数据大小和文件数量
- ✅ 检查是否包含必需的 GeoTIFF 文件
- ✅ 确认数据是否准备好进行预处理

**下一步**：如果所有检查通过，可以开始运行数据生成流程！